# WO-03 Q2 — flow relation of neighbours
Class each ring neighbour as upstream of the core, downstream of it, or off its stream, from `next_down` (and `main_bas`) in `basin06`. Then: how the core's rank on the `_upstream` variables depends on that class at Santa Fe, Timbuktu and Kaifeng; and how often a ring holds at least one upstream and one downstream neighbour in the WO-02 sample. Needs `output/edop/whg/q1_pairs.csv` (Q1 notebook, Cell 12). Read-only.

In [1]:
%matplotlib inline
# Cell 1 — Setup; load the pair table and the L6 topology
import sys, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.areas import engine as E
from scripts.edop.whg import wo03_common as C

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
topo = pd.DataFrame(conn.execute(
    "SELECT hybas_id::bigint, next_down::bigint, main_bas::bigint, sub_area, up_area FROM public.basin06").fetchall(),
    columns=['hybas_id', 'next_down', 'main_bas', 'sub_area', 'up_area']).set_index('hybas_id')
ND = topo['next_down'].to_dict()
print('commit', C.commit(), '| pairs:', len(PAIRS), PAIRS['source'].value_counts().to_dict(), '| basins in topology:', len(topo))
print('next_down = 0 (no next basin):', int((topo['next_down'] == 0).sum()), '| next_down not found among basin ids:', int((~topo['next_down'].isin(topo.index) & (topo['next_down'] != 0)).sum()))

commit 2c9a486 | pairs: 1739 {'sample': 1680, 'test': 59} | basins in topology: 16397
next_down = 0 (no next basin): 5149 | next_down not found among basin ids: 0


In [2]:
# Cell 2 — Classify every neighbour by following next_down from each end
# up1 / upN: the neighbour is upstream of the core (core is its next_down, or further down its chain, N steps)
# dn1 / dnN: the neighbour is downstream of the core (it is the core's next_down, or further down the core's chain)
# same / other: off the core's stream; same or different main_bas
def chain(h, limit=5000):
    out, seen = [], set()
    while True:
        n = ND.get(h, 0)
        if n == 0 or n in seen or len(out) >= limit:
            return out
        out.append(n); seen.add(n); h = n
def classify(core, nb):
    cc, nc = chain(core), chain(nb)
    if nb in cc:
        k = cc.index(nb) + 1
        return ('dn1' if k == 1 else 'dnN'), k
    if core in nc:
        k = nc.index(core) + 1
        return ('up1' if k == 1 else 'upN'), k
    return ('same' if topo.at[core, 'main_bas'] == topo.at[nb, 'main_bas'] else 'other'), 0
res = PAIRS.apply(lambda r: classify(int(r['core_id']), int(r['nb_id'])), axis=1)
PAIRS['flow'] = [r[0] for r in res]; PAIRS['steps'] = [r[1] for r in res]
PAIRS['nb_up_area'] = PAIRS['nb_id'].map(topo['up_area']); PAIRS['core_up_area'] = PAIRS['core_id'].map(topo['up_area'])
PAIRS.to_csv(C.OUT / 'q2_flow_pairs.csv', index=False)
ORDER = ['up1', 'upN', 'dn1', 'dnN', 'same', 'other']
tab = pd.crosstab(PAIRS['source'], PAIRS['flow']).reindex(columns=ORDER, fill_value=0)
tab['all'] = tab.sum(axis=1)
L = ['neighbours by flow class:', tab.to_string(), '']
L.append('as % of neighbours:')
L.append((100 * tab[ORDER].div(tab['all'], axis=0)).round(1).to_string())
L.append('')
L.append('test places (neighbours by class):')
tt = pd.crosstab(PAIRS.loc[PAIRS.source == 'test', 'tag'], PAIRS.loc[PAIRS.source == 'test', 'flow']).reindex(columns=ORDER, fill_value=0)
L.append(tt.to_string())
print('\n'.join(L))

neighbours by flow class:
flow    up1  upN  dn1  dnN  same  other   all
source                                       
sample  182   26  173   27   467    805  1680
test      4    0    7    1    24     23    59

as % of neighbours:
flow     up1  upN   dn1  dnN  same  other
source                                   
sample  10.8  1.5  10.3  1.6  27.8   47.9
test     6.8  0.0  11.9  1.7  40.7   39.0

test places (neighbours by class):
flow          up1  upN  dn1  dnN  same  other
tag                                          
Alexandria      0    0    0    0     0      5
Chaco Canyon    0    0    1    0     5      1
Kaifeng         0    0    1    1     4      1
Lviv            0    0    1    0     5      6
Poltava         0    0    1    0     2      1
Santa Fe        2    0    1    0     4      0
Tbilisi         0    0    1    0     3      5
Timbuktu        2    0    1    0     1      1
Utqiagvik       0    0    0    0     0      3


In [3]:
# Cell 3 — Sample: how often does a ring hold at least one upstream and one downstream neighbour?  Plus topology sanity checks
S = PAIRS[PAIRS.source == 'sample']
g = S.groupby('core_id')['flow'].agg(lambda s: set(s))
n_rings = len(g)
has_up = g.apply(lambda s: bool(s & {'up1', 'upN'})); has_dn = g.apply(lambda s: bool(s & {'dn1', 'dnN'}))
rows = [dict(ring_contains='at least one upstream', rings=int(has_up.sum())),
        dict(ring_contains='at least one downstream', rings=int(has_dn.sum())),
        dict(ring_contains='both', rings=int((has_up & has_dn).sum())),
        dict(ring_contains='neither (all off the stream)', rings=int((~has_up & ~has_dn).sum())),
        dict(ring_contains='direct upstream (up1)', rings=int(g.apply(lambda s: 'up1' in s).sum())),
        dict(ring_contains='direct downstream (dn1)', rings=int(g.apply(lambda s: 'dn1' in s).sum()))]
T = pd.DataFrame(rows); T['pct_of_rings'] = (100 * T['rings'] / n_rings).round(1)
L = [f'sample rings with at least one neighbour: {n_rings} (of 306 sample cores)', T.to_string(index=False), '']
cores = PAIRS.drop_duplicates('core_id')
nd_cores = cores[cores['core_id'].map(ND) != 0]
inring = PAIRS.groupby('core_id')['nb_id'].agg(set)
miss = [c for c in nd_cores['core_id'] if ND[c] not in inring.get(c, set())]
L.append(f'cores with a next_down basin: {len(nd_cores)} | next_down basin NOT among the ring neighbours: {len(miss)} ({100 * len(miss) / max(len(nd_cores), 1):.1f}%)')
L.append('steps to the neighbour along the chain (upN and dnN):')
L.append(PAIRS.loc[PAIRS.flow.isin(['upN', 'dnN']), ['flow', 'steps']].groupby('flow')['steps'].describe().round(1).to_string())
byc = S.groupby('core_id')['flow'].apply(lambda s: s.isin(['same', 'other']).all())
L.append(f'sample rings where every neighbour is off the stream: {int(byc.sum())} of {n_rings}')
print('\n'.join(L))

sample rings with at least one neighbour: 298 (of 306 sample cores)
               ring_contains  rings  pct_of_rings
       at least one upstream     91          30.5
     at least one downstream    173          58.1
                        both     80          26.8
neither (all off the stream)    114          38.3
       direct upstream (up1)     91          30.5
     direct downstream (dn1)    173          58.1

cores with a next_down basin: 185 | next_down basin NOT among the ring neighbours: 5 (2.7%)
steps to the neighbour along the chain (upN and dnN):
      count  mean  std  min  25%  50%  75%   max
flow                                            
dnN    28.0   2.5  1.9  2.0  2.0  2.0  2.0  12.0
upN    26.0   2.5  1.0  2.0  2.0  2.0  2.8   6.0
sample rings where every neighbour is off the stream: 114 of 298


In [6]:
# Cell 4 — Santa Fe, Timbuktu, Kaifeng: the core's rank on each _upstream variable, with all neighbours and with off-stream neighbours only
meta = E.load_catalog(level=6)
UPV = [v for v in meta.index if v.endswith('_upstream') and meta.at[v, 'kind'] == 'continuous']
print('_upstream continuous variables:', len(UPV), UPV)
def bset(ids):
    return pd.DataFrame({'hybas_id': np.array(ids, dtype='int64'), 'weight': 1.0})
def rank_str(scores, core):
    s = scores.dropna()
    if core not in s.index:
        return 'n/a'
    return f"{1 + int((s > s[core]).sum())}/{len(s)}"
blocks = []
for place in ['Santa Fe', 'Timbuktu', 'Kaifeng']:
    P = PAIRS[(PAIRS.source == 'test') & (PAIRS.tag == place)].sort_values('flow', key=lambda c: c.map({k: i for i, k in enumerate(ORDER)}))
    core = int(P['core_id'].iloc[0])
    ids = [core] + [int(x) for x in P['nb_id']]
    m = E.attach_values(bset(ids), meta, conn, E._LEVEL_TABLE[6], E._LEVEL_VIEW[6])[0]
    sc = m.loc[ids, UPV].astype(float)
    lab = {core: 'CORE'}; lab.update({int(r.nb_id): f"{r.flow}:{str(int(r.nb_id))[-5:]}" for r in P.itertuples()})
    off = [core] + [int(r.nb_id) for r in P.itertuples() if r.flow in ('same', 'other')]
    out = sc.T.rename(columns=lab).round(0)
    out['core_rank_all'] = [rank_str(sc[v], core) for v in UPV]
    out['core_rank_offstream'] = [rank_str(sc.loc[off, v], core) if len(off) > 1 else 'n/a' for v in UPV]
    ndiff = sum(1 for v in UPV if rank_str(sc[v], core) != (rank_str(sc.loc[off, v], core) if len(off) > 1 else 'n/a'))
    up_a = {core: topo.at[core, 'up_area']}; up_a.update({int(r.nb_id): topo.at[int(r.nb_id), 'up_area'] for r in P.itertuples()})
    blocks.append(f"{place}: core {core}, up_area {topo.at[core, 'up_area']:.0f} km2; neighbours by class: {P['flow'].value_counts().to_dict()}; "
                  f"up_area by basin: {({lab[k]: round(v) for k, v in up_a.items()})}\n"
                  f"variables whose rank changes when only off-stream neighbours are kept: {ndiff} of {len(UPV)}\n" + out.to_string())
print('\n\n'.join(blocks))

Santa Fe: core 7060674070, up_area 49334 km2; neighbours by class: {'same': 4, 'up1': 2, 'dn1': 1}; up_area by basin: {'CORE': 49334, 'up1:36480': 26441, 'up1:36470': 8235, 'dn1:75990': 66238, 'same:90100': 21967, 'same:74060': 18885, 'same:74010': 5829, 'same:74140': 5264}
variables whose rank changes when only off-stream neighbours are kept: 14 of 14
hybas_id                     CORE  up1:36480  up1:36470  dn1:75990  same:90100  same:74060  same:74010  same:74140 core_rank_all core_rank_offstream
aridity_upstream             34.0       40.0       36.0       31.0        24.0        22.0        24.0        20.0           3/8                 1/5
cropland_extent_upstream     32.0       43.0       10.0       28.0        10.0         0.0        28.0         0.0           2/8                 1/5
human_footprint_09_upstream  53.0       41.0       40.0       51.0        47.0        44.0        60.0        32.0           2/8                 2/5
karst_upstream               43.0       30.0     

In [4]:
# Cell 5 — The cores whose next_down basin is not a ring neighbour; and does a ring hold every direct upstream basin of its core?
cores = PAIRS.drop_duplicates('core_id')[['core_id', 'source', 'tag']]
inring = PAIRS.groupby('core_id')['nb_id'].agg(set)
rows = []
for r in cores.itertuples():
    nd = ND[r.core_id]
    if nd == 0 or nd in inring.get(r.core_id, set()):
        continue
    d = conn.execute(
        "SELECT ST_Distance(a.geom::geography, b.geom::geography), a.endo, a.coast, a.sub_area, b.sub_area, b.endo, b.coast "
        "FROM public.basin06 a, public.basin06 b WHERE a.hybas_id = %s AND b.hybas_id = %s", (r.core_id, nd)).fetchone()
    rows.append(dict(core_id=r.core_id, source=r.source, next_down=nd, next_down_is_main_bas=(nd == int(topo.at[r.core_id, 'main_bas'])),
                     core_endo=d[1], core_coast=d[2], core_km2=round(d[3], 1), nd_km2=round(d[4], 1), nd_endo=d[5], nd_coast=d[6], gap_m=round(d[0])))
L = ['cores whose next_down basin is not among the ring neighbours:']
L.append(pd.DataFrame(rows).to_string(index=False) if rows else 'none')
L.append('')
ups = topo.reset_index().groupby('next_down')['hybas_id'].agg(list)
U = pd.DataFrame([dict(core_id=r.core_id, source=r.source, n_up_all=len(ups.get(r.core_id, [])),
                       n_up_in_ring=len(set(ups.get(r.core_id, [])) & inring.get(r.core_id, set()))) for r in cores.itertuples()])
has = U[U['n_up_all'] > 0]
L.append(f"cores: {len(U)} | with at least one basin whose next_down is the core: {len(has)}")
L.append(f"direct upstream basins in total: {int(has['n_up_all'].sum())} | of them in the ring: {int(has['n_up_in_ring'].sum())}")
L.append(f"cores whose ring holds all their direct upstream basins: {int((has['n_up_all'] == has['n_up_in_ring']).sum())} | missing at least one: {int((has['n_up_all'] > has['n_up_in_ring']).sum())}")
miss = has.assign(missing=has['n_up_all'] - has['n_up_in_ring'])
L.append('direct upstream basins missing from the ring, per core (cores with at least one missing):')
L.append(miss.loc[miss['missing'] > 0, 'missing'].describe().round(1).to_string() if (miss['missing'] > 0).any() else 'none')
print('\n'.join(L))

cores whose next_down basin is not among the ring neighbours:
   core_id source  next_down  next_down_is_main_bas  core_endo  core_coast  core_km2  nd_km2  nd_endo  nd_coast  gap_m
3060024470 sample 3060496490                  False          2           0   22151.2 19821.1        0         0 243638
2060086960 sample 2060066340                   True          2           0   23752.4  3193.4        0         0  28949
6060041060 sample 6060018850                   True          2           0    5213.6 27381.9        0         0 188396
7060075100 sample 7060049270                   True          2           0    6437.8  6913.5        0         0 340036
5060091190 sample 5060491370                  False          2           0    1863.6   141.9        1         0 432483

cores: 307 | with at least one basin whose next_down is the core: 93
direct upstream basins in total: 186 | of them in the ring: 186
cores whose ring holds all their direct upstream basins: 93 | missing at least one: 0
dire

In [5]:
# Cell 6 — Virtual connections cut: a sink (endo = 2) with next_down > 0 is a topological link, not surface flow (source documentation for next_down)
# Re-class every pair following only real links; compare with Cell 2's classes. "Same system" is now: both chains end at the same basin.
endo = pd.Series(dict(conn.execute("SELECT hybas_id::bigint, endo FROM public.basin06").fetchall()))
virtual = {h for h in topo.index if endo.get(h, 0) == 2 and ND[h] > 0}
print('basins with a virtual next_down (endo = 2 and next_down > 0):', len(virtual))

def chain2(h, real_only, limit=5000):
    nodes, seen, via_v = [], set(), False
    while True:
        n = ND.get(h, 0)
        if n == 0 or n in seen or len(nodes) >= limit:
            break
        v = h in virtual
        if v and real_only:
            break
        via_v = via_v or v
        nodes.append((n, via_v)); seen.add(n); h = n
    return nodes
def terminal(h, real_only):
    ch = chain2(h, real_only)
    return ch[-1][0] if ch else h
def classify2(core, nb, real_only):
    cc = {n: (i + 1, v) for i, (n, v) in enumerate(chain2(core, real_only))}
    nc = {n: (i + 1, v) for i, (n, v) in enumerate(chain2(nb, real_only))}
    if nb in cc:
        k, v = cc[nb]; return ('dn1' if k == 1 else 'dnN'), v
    if core in nc:
        k, v = nc[core]; return ('up1' if k == 1 else 'upN'), v
    if real_only:
        return ('same' if terminal(core, True) == terminal(nb, True) else 'other'), False
    return ('same' if topo.at[core, 'main_bas'] == topo.at[nb, 'main_bas'] else 'other'), False
a = PAIRS.apply(lambda r: classify2(int(r['core_id']), int(r['nb_id']), False), axis=1)
b = PAIRS.apply(lambda r: classify2(int(r['core_id']), int(r['nb_id']), True), axis=1)
PAIRS['flow_all'] = [x[0] for x in a]; PAIRS['uses_virtual'] = [x[1] for x in a]; PAIRS['flow_real'] = [x[0] for x in b]
PAIRS.to_csv(C.OUT / 'q2_flow_pairs.csv', index=False)

L = [f"check: Cell 2 classes reproduced by following all links: {bool((PAIRS['flow'] == PAIRS['flow_all']).all())}", '']
L.append('pairs whose class (all links followed) rests on at least one virtual link, by class:')
L.append(PAIRS.loc[PAIRS['uses_virtual'], 'flow_all'].value_counts().reindex(ORDER, fill_value=0).to_string())
L.append('')
L.append('class with all links (rows) versus class with virtual links cut (columns):')
L.append(pd.crosstab(PAIRS['flow_all'], PAIRS['flow_real']).reindex(index=ORDER, columns=ORDER, fill_value=0).to_string())
L.append('')
S = PAIRS[PAIRS.source == 'sample']
rows = []
for col in ['flow_all', 'flow_real']:
    g = S.groupby('core_id')[col].agg(lambda s: set(s)); n = len(g)
    up = g.apply(lambda s: bool(s & {'up1', 'upN'})); dn = g.apply(lambda s: bool(s & {'dn1', 'dnN'}))
    rows.append(dict(classes=col, rings=n, at_least_one_up=int(up.sum()), at_least_one_down=int(dn.sum()), both=int((up & dn).sum()), neither=int((~up & ~dn).sum())))
R = pd.DataFrame(rows)
for c in ['at_least_one_up', 'at_least_one_down', 'both', 'neither']:
    R[c + '_pct'] = (100 * R[c] / R['rings']).round(1)
L.append('sample rings: all links followed versus virtual links cut')
L.append(R.to_string(index=False))
print('\n'.join(L))

basins with a virtual next_down (endo = 2 and next_down > 0): 319
check: Cell 2 classes reproduced by following all links: True

pairs whose class (all links followed) rests on at least one virtual link, by class:
flow_all
up1      0
upN      0
dn1      0
dnN      0
same     0
other    0

class with all links (rows) versus class with virtual links cut (columns):
flow_real  up1  upN  dn1  dnN  same  other
flow_all                                  
up1        186    0    0    0     0      0
upN          0   26    0    0     0      0
dn1          0    0  180    0     0      0
dnN          0    0    0   28     0      0
same         0    0    0    0   461     30
other        0    0    0    0     0    828

sample rings: all links followed versus virtual links cut
  classes  rings  at_least_one_up  at_least_one_down  both  neither  at_least_one_up_pct  at_least_one_down_pct  both_pct  neither_pct
 flow_all    298               91                173    80      114                 30.5         

In [7]:
# Cell 7 — Three places: how far each neighbour's _upstream scores are from the core's, by flow class (virtual links cut) and catchment-size ratio
# Uses meta, UPV and bset from Cell 4, flow_real from Cell 6. Difference = |core score - neighbour score| in percentile points, per variable.
rows = []
for place in ['Santa Fe', 'Timbuktu', 'Kaifeng']:
    P = PAIRS[(PAIRS.source == 'test') & (PAIRS.tag == place)]
    core = int(P['core_id'].iloc[0]); ids = [core] + [int(x) for x in P['nb_id']]
    m = E.attach_values(bset(ids), meta, conn, E._LEVEL_TABLE[6], E._LEVEL_VIEW[6])[0]
    sc = m.loc[ids, UPV].astype(float)
    for r in P.itertuples():
        a, b = sc.loc[core].to_numpy(), sc.loc[int(r.nb_id)].to_numpy()
        d = np.abs(a - b); both0 = (a == 0) & (b == 0)
        dd = pd.Series(d); de = pd.Series(np.where(both0, np.nan, d))
        rows.append(dict(place=place, nb=str(int(r.nb_id))[-5:], flow=r.flow_real, up_area_ratio=round(r.nb_up_area / r.core_up_area, 2),
                         median_abs_diff=round(dd.median(), 1), median_excl_both0=round(de.median(), 1),
                         max_abs_diff=round(dd.max(), 1), n_both0=int(both0.sum())))
T3 = pd.DataFrame(rows)
T3['flow'] = pd.Categorical(T3['flow'], categories=ORDER, ordered=True)
print('percentile-point difference between core and neighbour over the', len(UPV), '_upstream variables; n_both0 = variables where both are 0')
print(T3.sort_values(['place', 'flow', 'up_area_ratio']).to_string(index=False))

percentile-point difference between core and neighbour over the 14 _upstream variables; n_both0 = variables where both are 0
   place    nb  flow  up_area_ratio  median_abs_diff  median_excl_both0  max_abs_diff  n_both0
 Kaifeng 70500   dn1           8.21              5.5                7.3          44.7        1
 Kaifeng 94190   dnN          11.27              5.0                6.6          42.1        1
 Kaifeng 56150  same           0.40              2.5                3.0          28.7        2
 Kaifeng 70370  same           0.86              1.6                3.3          17.4        3
 Kaifeng 76710  same           2.10              1.5                2.5          51.0        2
 Kaifeng 76030  same           7.01              6.2                8.0          49.5        1
 Kaifeng 79370 other          48.46             28.1               29.2          85.1        2
Santa Fe 36470   up1           0.17              5.4                5.4          34.0        0
Santa Fe 36480   up1

In [8]:
# Cell 8 — Whole sample: the same difference for every pair, by flow class (virtual links cut) and catchment-size ratio
# One attach_values call over every core and neighbour (set route verified in Q1), then differences in numpy. Uses meta, UPV, bset from Cell 4.
import time
ids = sorted(set(PAIRS['core_id']) | set(PAIRS['nb_id']))
t0 = time.perf_counter()
m = E.attach_values(bset(ids), meta, conn, E._LEVEL_TABLE[6], E._LEVEL_VIEW[6])[0]
print(f'attach_values over {len(ids)} basins: {time.perf_counter() - t0:.1f}s')
SC = m[UPV].astype(float)
A = SC.loc[PAIRS['core_id'].to_numpy()].to_numpy(); Bm = SC.loc[PAIRS['nb_id'].to_numpy()].to_numpy()
D = np.abs(A - Bm)
both0 = (A == 0) & (Bm == 0)
De = np.where(both0, np.nan, D)
PAIRS['n_valid'] = np.sum(~np.isnan(De), axis=1)
PAIRS['med_diff'] = pd.DataFrame(De).median(axis=1).to_numpy()      # median over variables, both-zero variables excluded
PAIRS['up_ratio'] = PAIRS['nb_up_area'] / PAIRS['core_up_area']
PAIRS.to_csv(C.OUT / 'q2_flow_pairs.csv', index=False)

K = PAIRS[PAIRS['n_valid'] >= 5].copy()
K['group'] = K['flow_real'].map({'up1': 'upstream', 'upN': 'upstream', 'dn1': 'downstream', 'dnN': 'downstream', 'same': 'off-stream, same system', 'other': 'off-stream, other system'})
G = ['upstream', 'downstream', 'off-stream, same system', 'off-stream, other system']
L = [f'pairs: {len(PAIRS)} | with at least 5 variables where not both are zero: {len(K)} (dropped {len(PAIRS) - len(K)})', '']
L.append('median over variables of |core - neighbour| (percentile points), by class; quartiles across pairs:')
L.append(K.groupby('group')['med_diff'].describe(percentiles=[.25, .5, .75]).reindex(G).round(1).to_string())
L.append('')
bins = [0, 0.5, 0.9, 1.1, 2, 10, 1e9]; labs = ['<0.5', '0.5-0.9', '0.9-1.1', '1.1-2', '2-10', '>10']
K['ratio_bin'] = pd.cut(K['up_ratio'], bins=bins, labels=labs)
L.append('median of the per-pair median difference, by class and neighbour/core catchment-area ratio (pairs in brackets):')
piv = K.pivot_table(index='group', columns='ratio_bin', values='med_diff', aggfunc='median', observed=False).reindex(G)
cnt = K.pivot_table(index='group', columns='ratio_bin', values='med_diff', aggfunc='count', observed=False).reindex(G)
L.append(piv.round(1).astype(object).where(piv.notna(), '-').combine(cnt, lambda s, c: s.astype(str) + ' (' + c.fillna(0).astype(int).astype(str) + ')').to_string())
print('\n'.join(L))

/var/folders/f9/r5mr431d23zcpktsjz_xjxt40000gn/T/ipykernel_84747/3845316883.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  L.append(piv.round(1).astype(object).where(piv.notna(), '-').combine(cnt, lambda s, c: s.astype(str) + ' (' + c.fillna(0).astype(int).astype(str) + ')').to_string())
/var/folders/f9/r5mr431d23zcpktsjz_xjxt40000gn/T/ipykernel_84747/3845316883.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  L.append(piv.round(1).astype(object).where(piv.notna(), '-').combine(cnt, lambda s, c: s.astype(str) + ' (' + c.fillna(0).astype(int).as

In [9]:
# Cell 9 — Cell 8's tables again, without the object-dtype formatting that raised the FutureWarning (reads n_valid, med_diff, up_ratio, flow_real from PAIRS)
print('columns present from Cell 8:', all(c in PAIRS.columns for c in ['n_valid', 'med_diff', 'up_ratio', 'flow_real']))
K = PAIRS[PAIRS['n_valid'] >= 5].copy()
K['group'] = K['flow_real'].map({'up1': 'upstream', 'upN': 'upstream', 'dn1': 'downstream', 'dnN': 'downstream',
                                 'same': 'off-stream, same system', 'other': 'off-stream, other system'})
G = ['upstream', 'downstream', 'off-stream, same system', 'off-stream, other system']
L = [f'pairs: {len(PAIRS)} | with at least 5 variables where not both are zero: {len(K)} (dropped {len(PAIRS) - len(K)})', '']
L.append('median over variables of |core - neighbour| (percentile points), by class; quartiles across pairs:')
L.append(K.groupby('group')['med_diff'].describe(percentiles=[.25, .5, .75]).reindex(G).round(1).to_string())
L.append('')
bins = [0, 0.5, 0.9, 1.1, 2, 10, 1e9]; labs = ['<0.5', '0.5-0.9', '0.9-1.1', '1.1-2', '2-10', '>10']
K['ratio_bin'] = pd.cut(K['up_ratio'], bins=bins, labels=labs)
rows = []
for gname in G:
    row = {'class': gname}
    for lab in labs:
        s = K.loc[(K['group'] == gname) & (K['ratio_bin'] == lab), 'med_diff']
        row[lab] = f"{s.median():.1f} ({len(s)})" if len(s) else '- (0)'
    rows.append(row)
L.append('median of the per-pair median difference, by class and neighbour/core catchment-area ratio (pairs in brackets):')
L.append(pd.DataFrame(rows).to_string(index=False))
print('\n'.join(L))

columns present from Cell 8: True
pairs: 1739 | with at least 5 variables where not both are zero: 1470 (dropped 269)

median over variables of |core - neighbour| (percentile points), by class; quartiles across pairs:
                          count  mean  std  min  25%  50%   75%   max
group                                                                
upstream                  174.0   5.0  5.1  0.0  1.4  3.5   7.0  25.8
downstream                167.0   4.3  4.3  0.0  1.4  3.4   6.0  26.2
off-stream, same system   393.0   8.7  5.8  0.4  4.3  7.5  11.7  33.0
off-stream, other system  736.0  11.2  7.9  0.5  5.3  9.1  15.1  48.6

median of the per-pair median difference, by class and neighbour/core catchment-area ratio (pairs in brackets):
                   class      <0.5   0.5-0.9  0.9-1.1    1.1-2      2-10       >10
                upstream 5.6 (111)  1.8 (44) 0.0 (19)    - (0)     - (0)     - (0)
              downstream     - (0)     - (0) 0.0 (15) 1.9 (54)  4.1 (68)  7.8 (30)
